# Getting Started with TAFS

This notebook shows how to import and use the `src` package interactively.

**Before running:** make sure you have activated the virtual environment and installed dependencies:
```bash
source .venv/bin/activate
make requirements
```

Jupyter runs from the project root, so `src` is directly importable — no install step needed.

In [ ]:
# Verify imports work.
from src.tafs.data.synthetic import SyntheticConfig, generate_dataset
from src.tafs.data.datamodule import TAFSSyntheticDataModule
from src.tafs.models.baselines import UniformEnsemble, OracleRouter

print('Imports OK')

## 1. Generate synthetic data

The synthetic dataset simulates a two-regime time series where different
base predictors are optimal in each regime. See `src/tafs/data/synthetic.py`.

In [ ]:
# Generate a small dataset in-memory (no caching).
cfg = SyntheticConfig(n_trajectories=50, T=400, seed=0)
data = generate_dataset(cfg)

print('Keys:', list(data.keys()))
print('feature_matrix shape:', data['feature_matrix'].shape)   # (N, p)
print('error_matrix shape:  ', data['error_matrix'].shape)    # (N, K)
print('regime distribution: ', {r: int((data['regime'] == r).sum()) for r in [0, 1]})

## 2. Build a DataModule

`TAFSSyntheticDataModule` wraps the generator with manifest-aware caching
and exposes `train_dataloader()` / `val_dataloader()` / `test_dataloader()`.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    dm = TAFSSyntheticDataModule(
        cache_dir=tmp + '/cache',
        n_trajectories=50,
        T=40,
        seed=0,
        batch_size=16,
    )
    dm.setup('fit')

    batch = next(iter(dm.train_dataloader()))
    for key, val in batch.items():
        print(f'  {key}: {val.shape} {val.dtype}')

## 3. Evaluate baselines

Once you implement `TAFS.forward()`, you can swap `UniformEnsemble` for `TAFS`
and call `model.evaluate(dm)` the same way.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    dm = TAFSSyntheticDataModule(
        cache_dir=tmp + '/cache',
        n_trajectories=100,
        T=40,
        seed=0,
        batch_size=32,
    )
    dm.setup('fit')

    for name, model in [
        ('Uniform', UniformEnsemble(num_experts=len(dm.expert_names))),
        ('Oracle',  OracleRouter(num_experts=len(dm.expert_names))),
    ]:
        metrics = model.evaluate(dm)
        print(f'{name:10s}  selected_error={metrics["selected_error_mean"]:.4f}  oracle={metrics["oracle_error_mean"]:.4f}')